<a href="https://colab.research.google.com/github/HopeSilkina/deposits_forecast_project/blob/main/notebooks/06_Final_Forecast_Deposits_2026_2027_Ru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# БЛОК 6: ОКОНЧАТЕЛЬНЫЙ ПРОГНОЗ НА 2026-2027 ГОДЫ
# Проект: Прогнозирование объема вкладов населения РФ
# Автор: Надежда Силкина
# Дата: 2026
# ============================================================

# ============================================================
# 1. ПОДКЛЮЧЕНИЕ БИБЛИОТЕК
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Настройка графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Библиотеки загружены")

# ============================================================
# 2. ЗАГРУЗКА ДАННЫХ
# ============================================================

url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

print(f"✅ Данные загружены. Записей: {len(df)}")
print(f"Период: с {df.index.min()} по {df.index.max()}")

# ============================================================
# 3. ВОССТАНОВЛЕНИЕ RIDGE-МОДЕЛИ (38 ПРИЗНАКОВ ИЗ БЛОКА 4)
# ============================================================

print("\n" + "="*60)
print("3. ВОССТАНОВЛЕНИЕ RIDGE-МОДЕЛИ (38 ПРИЗНАКОВ)")
print("="*60)

# Создаем копию для добавления признаков
df_model = df.copy()

# 3.1. Лаги DEPOS
df_model['DEPOS_log'] = np.log(df_model['DEPOS'])
for lag in [1, 3, 6, 12]:
    df_model[f'DEPOS_lag_{lag}'] = df_model['DEPOS'].shift(lag)

# 3.2. Сезонные фиктивные переменные
df_model['Month'] = df_model.index.month
for month in range(2, 13):
    df_model[f'month_{month}'] = (df_model['Month'] == month).astype(int)

# 3.3. Структурные переменные
df_model['post_2022'] = (df_model.index >= '2023-01-01').astype(int)
df_model['covid'] = ((df_model.index >= '2020-03-01') & (df_model.index <= '2022-01-01')).astype(int)

# 3.4. Аномалии WAGE (адаптивный порог -2σ)
model_wage = LinearRegression()
model_wage.fit(df_model[['WAGE']].values, df_model['DEPOS'].values)
df_model['residual_wage'] = df_model['DEPOS'] - model_wage.predict(df_model[['WAGE']].values)
threshold_anomaly = -2.0 * df_model['residual_wage'].std()
df_model['anomaly_wage'] = (df_model['residual_wage'] < threshold_anomaly).astype(int)

# 3.5. Лаги макрофакторов
for col in ['WAGE', 'CPI', 'USDind']:
    for lag in [1, 3, 6]:
        df_model[f'{col}_lag_{lag}'] = df_model[col].shift(lag)

# 3.6. Взаимодействие UNEM × DEP1
df_model['UNEM_DEP1'] = df_model['UNEM'] * df_model['DEP1']

# Итоговый набор признаков (как в блоке 4)
feature_columns = [
    'WAGE', 'SERV', 'DEP1', 'CRED1', 'CPI', 'USDind', 'UNEM', 'IPI', 'IMP',
    'DEPOS_lag_1', 'DEPOS_lag_3', 'DEPOS_lag_6', 'DEPOS_lag_12',
    'month_2', 'month_3', 'month_4', 'month_5', 'month_6',
    'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12',
    'post_2022', 'covid', 'anomaly_wage',
    'WAGE_lag_1', 'WAGE_lag_3', 'WAGE_lag_6',
    'CPI_lag_1', 'CPI_lag_3', 'CPI_lag_6',
    'USDind_lag_1', 'USDind_lag_3', 'USDind_lag_6',
    'UNEM_DEP1'
]

print(f"📊 Всего признаков: {len(feature_columns)}")

# Подготовка данных для обучения
X = df_model[feature_columns].dropna()
y = df_model.loc[X.index, 'DEPOS']

print(f"📊 Наблюдений с полными данными: {len(X)}")
print(f"📊 Период: {X.index[0].strftime('%Y-%m')} — {X.index[-1].strftime('%Y-%m')}")

# Масштабирование
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Обучение модели на ВСЕХ данных (134 наблюдения)
ridge_final = Ridge(alpha=1.0)
ridge_final.fit(X_scaled, y)

# Проверка качества на обучающей выборке
y_pred_train = ridge_final.predict(X_scaled)
r2_train = r2_score(y, y_pred_train)
rmse_train = np.sqrt(mean_squared_error(y, y_pred_train))
mae_train = mean_absolute_error(y, y_pred_train)

print(f"\n✅ Ridge-модель обучена на всех данных (n={len(X)})")
print(f"   R²_train = {r2_train:.4f}")
print(f"   RMSE_train = {rmse_train:.2f} млрд руб.")
print(f"   MAE_train = {mae_train:.2f} млрд руб.")
print(f"\n📌 ПРИМЕЧАНИЕ: Модель переобучена на всех 134 наблюдениях для")
print(f"   максимального использования информации. Структура идентична")
print(f"   проверенной модели (R²_test = 0.9422 на тестовой выборке).")

# ============================================================
# 4. ПОДГОТОВКА СЦЕНАРИЕВ ДЛЯ МАКРОФАКТОРОВ
# ============================================================

print("\n" + "="*60)
print("4. ПОДГОТОВКА СЦЕНАРИЕВ НА 12 МЕСЯЦЕВ (март 2026 — февраль 2027)")
print("="*60)

# Последняя фактическая дата
last_date = df.index[-1]
print(f"\nПоследняя фактическая дата: {last_date.strftime('%Y-%m-%d')}")

# Будущие даты: МАРТ 2026 — ФЕВРАЛЬ 2027 (12 месяцев)
future_dates = pd.date_range(start='2026-03-31', periods=12, freq='ME')
print(f"Период прогноза: {future_dates[0].strftime('%Y-%m')} — {future_dates[-1].strftime('%Y-%m')}")

# Последние известные значения (февраль 2026)
last_values = df.iloc[-1]
last_depos = last_values['DEPOS']

print(f"\nТекущий уровень DEPOS (февраль 2026): {last_depos:,.0f} млрд руб.")

# Функция для создания сценария
def create_scenario(growth_params, scenario_name):
    """
    Создает сценарий для макрофакторов на 12 месяцев.

    growth_params: dict с параметрами роста/изменения для каждого фактора
    """
    scenario = pd.DataFrame(index=future_dates)

    # WAGE: рост с учетом сценария
    last_wage = last_values['WAGE']
    scenario['WAGE'] = [last_wage * (1 + growth_params['WAGE']) ** (i+1) for i in range(12)]

    # SERV: рост
    last_serv = last_values['SERV']
    scenario['SERV'] = [last_serv * (1 + growth_params['SERV']) ** (i+1) for i in range(12)]

    # DEP1: уровень ставки
    scenario['DEP1'] = last_values['DEP1'] * (1 + growth_params['DEP1'])

    # CRED1
    scenario['CRED1'] = last_values['CRED1'] * (1 + growth_params['CRED1'])

    # CPI: уровень
    scenario['CPI'] = last_values['CPI'] + growth_params['CPI']

    # USDind: волатильность
    scenario['USDind'] = growth_params['USDind']

    # UNEM: уровень
    scenario['UNEM'] = last_values['UNEM'] * (1 + growth_params['UNEM'])

    # IPI: индекс
    scenario['IPI'] = last_values['IPI'] * (1 + growth_params['IPI'])

    # IMP: импорт
    scenario['IMP'] = last_values['IMP'] * (1 + growth_params['IMP'])

    # Структурные переменные
    scenario['post_2022'] = 1
    scenario['covid'] = 0
    scenario['anomaly_wage'] = 0

    # Взаимодействие
    scenario['UNEM_DEP1'] = scenario['UNEM'] * scenario['DEP1']

    # Инициализируем лаги DEPOS и макрофакторов
    for col in ['DEPOS_lag_1', 'DEPOS_lag_3', 'DEPOS_lag_6', 'DEPOS_lag_12']:
        scenario[col] = np.nan

    for col in ['WAGE', 'CPI', 'USDind']:
        for lag in [1, 3, 6]:
            scenario[f'{col}_lag_{lag}'] = np.nan

    return scenario

# Определяем сценарии
scenarios = {
    'Базовый': {
        'WAGE': 0.007,    # +0.7% в месяц (~8.7% годовых)
        'SERV': 0.005,    # +0.5% в месяц
        'DEP1': -0.02,    # -2% (снижение ставки)
        'CRED1': -0.01,   # -1%
        'CPI': -0.02,     # -0.02 п.п. (инфляция снижается)
        'USDind': 0.3,    # умеренная волатильность
        'UNEM': 0.01,     # +1% (небольшой рост)
        'IPI': 0.002,     # +0.2% в месяц
        'IMP': 0.003      # +0.3% в месяц
    },
    'Оптимистичный': {
        'WAGE': 0.01,     # +1% в месяц (~12.7% годовых)
        'SERV': 0.008,    # +0.8% в месяц
        'DEP1': -0.05,    # -5% (снижение ставки)
        'CRED1': -0.03,   # -3%
        'CPI': -0.05,     # -0.05 п.п. (инфляция быстро снижается)
        'USDind': 0.1,    # низкая волатильность
        'UNEM': -0.02,    # -2% (снижение безработицы)
        'IPI': 0.005,     # +0.5% в месяц
        'IMP': 0.006      # +0.6% в месяц
    },
    'Пессимистичный': {
        'WAGE': 0.004,    # +0.4% в месяц (~4.9% годовых)
        'SERV': 0.002,    # +0.2% в месяц
        'DEP1': 0.05,     # +5% (рост ставки)
        'CRED1': 0.03,    # +3%
        'CPI': 0.05,      # +0.05 п.п. (инфляция растет)
        'USDind': 0.8,    # высокая волатильность
        'UNEM': 0.05,     # +5% (рост безработицы)
        'IPI': -0.003,    # -0.3% в месяц
        'IMP': -0.005     # -0.5% в месяц
    }
}

print("\n📊 Сценарии макрофакторов:")
for name, params in scenarios.items():
    print(f"\n   {name}:")
    print(f"     WAGE: {params['WAGE']*100:+.1f}%/мес")
    print(f"     CPI: {params['CPI']:+.2f} п.п.")
    print(f"     UNEM: {params['UNEM']*100:+.1f}%")
    print(f"     DEP1: {params['DEP1']*100:+.1f}%")

# ============================================================
# 5. ПРОГНОЗ НА 12 МЕСЯЦЕВ
# ============================================================

print("\n" + "="*60)
print("5. ПРОГНОЗ НА 12 МЕСЯЦЕВ (март 2026 — февраль 2027)")
print("="*60)

# Функция для прогноза по сценарию
def forecast_scenario(growth_params, scenario_name):
    """
    Итеративный прогноз на 12 месяцев.
    Лаги DEPOS обновляются на каждом шаге.
    """
    scenario = create_scenario(growth_params, scenario_name)

    predictions = []
    current_depos = last_depos

    # История DEPOS для лагов (фактические значения)
    dep_history = df['DEPOS'].tolist()

    for i, date in enumerate(future_dates):
        # Лаги DEPOS: используем фактические, где возможно, затем прогнозные
        scenario.loc[date, 'DEPOS_lag_1'] = dep_history[-1]
        scenario.loc[date, 'DEPOS_lag_3'] = dep_history[-3] if len(dep_history) >= 3 else dep_history[0]
        scenario.loc[date, 'DEPOS_lag_6'] = dep_history[-6] if len(dep_history) >= 6 else dep_history[0]
        scenario.loc[date, 'DEPOS_lag_12'] = dep_history[-12] if len(dep_history) >= 12 else dep_history[0]

        # Сезонные переменные
        month_num = date.month
        for m in range(2, 13):
            scenario.loc[date, f'month_{m}'] = 1 if month_num == m else 0

        # Лаги макрофакторов
        for col in ['WAGE', 'CPI', 'USDind']:
            # Лаг 1: предыдущий месяц (фактический или прогнозный)
            if i == 0:
                scenario.loc[date, f'{col}_lag_1'] = df[col].iloc[-1]
            else:
                scenario.loc[date, f'{col}_lag_1'] = scenario.loc[future_dates[i-1], col]

            # Лаг 3: 3 месяца назад
            if i < 3:
                scenario.loc[date, f'{col}_lag_3'] = df[col].iloc[-3+i]
            else:
                scenario.loc[date, f'{col}_lag_3'] = scenario.loc[future_dates[i-3], col]

            # Лаг 6: 6 месяцев назад
            if i < 6:
                scenario.loc[date, f'{col}_lag_6'] = df[col].iloc[-6+i]
            else:
                scenario.loc[date, f'{col}_lag_6'] = scenario.loc[future_dates[i-6], col]

        # Формируем вектор признаков
        X_pred = scenario.loc[date, feature_columns].values.reshape(1, -1)
        X_pred_scaled = scaler.transform(X_pred)

        # Прогноз
        pred = ridge_final.predict(X_pred_scaled)[0]
        predictions.append(pred)

        # Обновляем историю
        dep_history.append(pred)

    scenario['DEPOS_forecast'] = predictions
    return scenario, predictions

# Прогнозы по всем сценариям
forecasts = {}
for name, params in scenarios.items():
    print(f"\n🔧 Прогноз по сценарию «{name}»...")
    scenario, predictions = forecast_scenario(params, name)
    forecasts[name] = {
        'scenario': scenario,
        'predictions': predictions
    }
    print(f"   Готово: {len(predictions)} значений")
    print(f"   Диапазон: {min(predictions):,.0f} — {max(predictions):,.0f} млрд руб.")

# ============================================================
# 6. ДОВЕРИТЕЛЬНЫЕ ИНТЕРВАЛЫ
# ============================================================

print("\n" + "="*60)
print("6. ДОВЕРИТЕЛЬНЫЕ ИНТЕРВАЛЫ")
print("="*60)

# Стандартное отклонение остатков на обучающей выборке
residuals_train = y - y_pred_train
std_residuals = residuals_train.std()

print(f"Стандартное отклонение остатков: {std_residuals:.2f} млрд руб.")
print(f"95% доверительный интервал: ±{1.96 * std_residuals:.2f} млрд руб.")

# Добавляем интервалы к базовому сценарию
base_predictions = np.array(forecasts['Базовый']['predictions'])
lower_bound = base_predictions - 1.96 * std_residuals
upper_bound = base_predictions + 1.96 * std_residuals

# ============================================================
# 7. ВИЗУАЛИЗАЦИЯ ПРОГНОЗА
# ============================================================

print("\n" + "="*60)
print("7. ВИЗУАЛИЗАЦИЯ ПРОГНОЗА")
print("="*60)

plt.figure(figsize=(16, 8))

# Исторические данные (последние 24 месяца для контекста)
history_start = df.index[-24]
plt.plot(df.loc[history_start:].index, df.loc[history_start:, 'DEPOS'],
         label='Фактические данные', color='#1f77b4', linewidth=2.5)

# Прогнозы по сценариям
colors = {'Базовый': '#2ca02c', 'Оптимистичный': '#ff7f0e', 'Пессимистичный': '#d62728'}

for name, forecast_data in forecasts.items():
    predictions = forecast_data['predictions']
    plt.plot(future_dates, predictions, label=f'Прогноз: {name}',
             color=colors[name], linestyle='--', linewidth=2)

# Доверительный интервал для базового сценария
plt.fill_between(future_dates, lower_bound, upper_bound,
                 alpha=0.2, color='#2ca02c', label='95% ДИ (базовый)')

# Вертикальная линия — начало прогноза
plt.axvline(x=last_date, color='gray', linestyle=':', alpha=0.5, label='Начало прогноза')

plt.title('Прогноз объема вкладов населения РФ\nМарт 2026 — Февраль 2027 (3 сценария)',
          fontsize=14)
plt.xlabel('Дата')
plt.ylabel('Объем вкладов, млрд руб.')
plt.legend(loc='upper left', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('06_final_forecast_2026_2027.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ График сохранен: 06_final_forecast_2026_2027.png")

# ============================================================
# 8. ИТОГОВАЯ ТАБЛИЦА ПРОГНОЗА
# ============================================================

print("\n" + "="*60)
print("8. ИТОГОВАЯ ТАБЛИЦА ПРОГНОЗА")
print("="*60)

# Создаем таблицу
forecast_table = pd.DataFrame({
    'Месяц': [d.strftime('%Y-%m') for d in future_dates],
    'Базовый': forecasts['Базовый']['predictions'],
    'Оптимистичный': forecasts['Оптимистичный']['predictions'],
    'Пессимистичный': forecasts['Пессимистичный']['predictions'],
    'Нижняя граница 95%': lower_bound,
    'Верхняя граница 95%': upper_bound
})

forecast_table = forecast_table.round(0)
print("\n📊 Прогноз объема вкладов (млрд руб.):")
print(forecast_table.to_string(index=False))

# Сохранение в CSV
forecast_table.to_csv('06_forecast_2026_2027.csv', index=False)
print("\n✅ Таблица сохранена: 06_forecast_2026_2027.csv")

# ============================================================
# 9. СВОДНАЯ СТАТИСТИКА
# ============================================================

print("\n" + "="*60)
print("9. СВОДНАЯ СТАТИСТИКА ПРОГНОЗА")
print("="*60)

print(f"\n📊 Текущий уровень (февраль 2026): {last_depos:,.0f} млрд руб.")

for name in ['Базовый', 'Оптимистичный', 'Пессимистичный']:
    pred = forecasts[name]['predictions']
    print(f"\n   {name} сценарий:")
    print(f"     Мин: {min(pred):,.0f} млрд руб. ({future_dates[np.argmin(pred)].strftime('%B %Y')})")
    print(f"     Макс: {max(pred):,.0f} млрд руб. ({future_dates[np.argmax(pred)].strftime('%B %Y')})")
    print(f"     Среднее: {np.mean(pred):,.0f} млрд руб.")
    print(f"     К февралю 2027: {pred[-1]:,.0f} млрд руб.")
    print(f"     Рост за 12 мес: {(pred[-1] - last_depos) / last_depos * 100:+.1f}%")

# ============================================================
# 10. ИТОГОВЫЙ ВЫВОД
# ============================================================

print("\n" + "="*60)
print("10. ИТОГОВЫЙ ВЫВОД")
print("="*60)

base_final = forecasts['Базовый']['predictions'][-1]
opt_final = forecasts['Оптимистичный']['predictions'][-1]
pes_final = forecasts['Пессимистичный']['predictions'][-1]

print(f"""
📌 ФИНАЛЬНЫЙ ПРОГНОЗ ОБЪЕМА ВКЛАДОВ НАСЕЛЕНИЯ РФ
   Период: март 2026 — февраль 2027

1. ТЕКУЩИЙ УРОВЕНЬ (февраль 2026): {last_depos:,.0f} млрд руб.

2. БАЗОВЫЙ СЦЕНАРИЙ:
   К февралю 2027 года: {base_final:,.0f} млрд руб.
   Рост за 12 месяцев: {(base_final - last_depos) / last_depos * 100:+.1f}%
   95% ДИ: [{lower_bound[-1]:,.0f} — {upper_bound[-1]:,.0f}] млрд руб.

3. ОПТИМИСТИЧНЫЙ СЦЕНАРИЙ:
   К февралю 2027 года: {opt_final:,.0f} млрд руб.
   Рост за 12 месяцев: {(opt_final - last_depos) / last_depos * 100:+.1f}%

4. ПЕССИМИСТИЧНЫЙ СЦЕНАРИЙ:
   К февралю 2027 года: {pes_final:,.0f} млрд руб.
   Рост за 12 месяцев: {(pes_final - last_depos) / last_depos * 100:+.1f}%

5. КЛЮЧЕВЫЕ ДРАЙВЕРЫ:
   - WAGE (зарплата): главный макроэкономический фактор
   - DEPOS_lag_1: инерция вкладов
   - post_2022: структурный сдвиг после 2022 года

6. МЕТОДОЛОГИЯ:
   - Модель: Ridge-регрессия (alpha=1.0), 37 признаков
   - Обучена на всех 134 наблюдениях (янв 2015 — фев 2026)
   - Структура идентична проверенной модели (R²_test = 0.9422)
   - Итеративный прогноз с обновлением лагов DEPOS

7. ОГРАНИЧЕНИЯ:
   - Прогноз основан на упрощенных сценариях макрофакторов
   - Не учитывает возможные шоки (кризисы, санкции)
   - Точность снижается с горизонтом прогноза
   - Модель на 134 наблюдениях не имеет независимой проверки

8. РЕКОМЕНДАЦИИ:
   - Использовать базовый сценарий для планирования
   - Отслеживать фактические значения WAGE, CPI, UNEM
   - Обновлять прогноз ежемесячно при поступлении новых данных
""")

print("✅ БЛОК 6 ЗАВЕРШЕН")
print("📌 ПРОЕКТ ПОЛНОСТЬЮ ЗАВЕРШЕН")